# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td>Leonardo</td></tr>
  <tr><td>Nachname:</td><td>Mininni</td></tr>
  <tr><td>Immatrikulationsnummer:</td><td>15-181-928</td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[09.11.2026] / Raum: [MU O2.013] / [08:00 – 12:00]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (cinema.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (cinema.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `cinema.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — nur zu verwenden, falls die PostgreSQL-Verbindung nicht funktioniert.

**Tabellen:** `movies`, `cinemas`, `staff`, `screenings`, `customers`, `tickets`

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `cinema_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [25]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


### Datenbankverbindung zu PostgreSQL

In [26]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Host:      ', host)
    print('Datenbank: ', connection.execute(text('SELECT current_database()')).scalar())
engine.dispose()

Host:       localhost
Datenbank:  postgres


### Daten nach PostgreSQL laden

In [27]:
# Tabellen aus der SQLite-Datei cinema.db nach PostgreSQL übertragen
tables = ['movies', 'cinemas', 'staff', 'screenings', 'customers', 'tickets']

sqlite_engine = create_engine('sqlite:///cinema.db')
engine = create_engine(db_connection_url)

for table in tables:
    df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
    df.to_sql(table, engine, if_exists='replace', index=False)
    print(f'{table:<12}{len(df):>5} Zeilen')

sqlite_engine.dispose()
engine.dispose()

movies         15 Zeilen
cinemas         6 Zeilen
staff          30 Zeilen
screenings    300 Zeilen
customers     505 Zeilen
tickets      2000 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwenden Sie für alle Aufgaben die
> SQLite-Datenbank `cinema.db`. Ersetzen Sie dazu in den Code-Zellen:
> ```python
> engine = create_engine(db_connection_url)
> ```
> durch:
>
> ```python
> engine = create_engine('sqlite:///cinema.db')
> ```
>
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Prüfung laufen unverändert
> in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `cinema.db` werden nicht gewertet.*

### A1 (4 Punkte)

Die Tabelle `screenings` enthält `base_price` und die Tabelle `tickets` enthält `paid_price`.

Ein Datenbankarchitekt sagt: *«`paid_price` ist redundant — der Preis steht bereits in `screenings.base_price`. Die Spalte sollte entfernt werden.»*

Hat er recht? Begründen Sie Ihre Antwort mit einem konkreten Szenario aus dem Kinobetrieb (z.B. Rabattaktionen, Gutscheine).

In [28]:
df = pd.read_sql_query('''
SELECT
    base_price,
    paid_price
FROM screenings
INNER JOIN tickets
    ON screenings.screeningid = tickets.screeningid;
''', con=engine)

df

,base_price,paid_price
0,22.0,22.0
1,14.5,14.5
2,22.0,18.7
3,20.0,20.0
4,20.0,20.0
...,...,...
1995,22.0,19.8
1996,12.0,10.2
1997,22.0,19.8
1998,22.0,19.8


**Ihre Antwort:**

> [Nein, paid_price ist nicht redundant, da base_price den regulären Ticketpreis und paid_price den tatsächlich bezahlten Preis darstellt, der beispielsweise aufgrund von Rabatten oder Gutscheinen abweichen kann.
Der INNER JOIN wird verwendet, weil nur Screenings mit tatsächlich verkauften Tickets relevant sind, um beide Preise zu vergleichen; ein LEFT JOIN würde zusätzlich Screenings ohne verkaufte Tickets anzeigen (paid_price = NULL).]

### A2 (4 Punkte)

Das Datenteam analysiert die Tabelle `customers` und findet folgende Werte in der Spalte `membership_type`:

```
Standard, standard, STANDARD, Standart, Premium, premium, Student, student, Senior
```

Beschreiben Sie das konkrete Datenproblem und nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable).

**Ihre Antwort:**

> [Data Quality Management Modell => Konsistenz]

### A3 (4 Punkte)

In `cinema.db` ist `paid_price` bei keinem Ticket höher als der `base_price` der zugehörigen Vorstellung. Angenommen, ein neues Ticket hätte `paid_price` = CHF 18.00 bei einem `base_price` von CHF 16.00: Wäre das **zwingend** ein Datenfehler? Begründen Sie in einem Satz.

**Ihre Antwort:**

> [Nein, wenn der effektiv bezahlte Preis höher als der Basispreis ist, muss das nicht zwingend ein Datenfehler sein (bspw. Dynamic Pricing)]

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `cinema.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl verkaufter Tickets pro Kino — für **alle** Kinos in der Datenbank.

```sql
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
JOIN tickets t ON s.screeningid = t.screeningid
JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name;
```

Die Tabelle `cinemas` enthält **6 Kinos**. Tatsächliches Ergebnis der SQL-Abfrage: nur **5 Zeilen**, Ticketanzahl zwischen 350 und 450.

Das Management fragt: *«Zeig mir auch Kinos, die noch keine Tickets verkauft haben.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 6 Kinos** erscheinen und ein Kino ohne Tickets korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [29]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query('''
SELECT c.name, COUNT(*) AS ticket_count
FROM screenings s
FULL OUTER JOIN tickets t ON s.screeningid = t.screeningid
FULL OUTER JOIN cinemas c ON s.cinemaid = c.cinemaid
GROUP BY c.name;
''', con=engine)

engine.dispose()
df_B1

,name,ticket_count
0,Rex Kino,387
1,Cineplexx,383
2,Kinok,441
3,Pathé Küchlin,436
4,Kino Aurora,1
5,Arthouse Metropol,354


### B2 (8 Punkte)

**Ziel:** Liste aller Kunden (`customerid`, `firstname`, `lastname`), die noch nie ein Ticket gekauft haben.

```sql
SELECT c.customerid, c.firstname, c.lastname
FROM customers c
JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
```

Tatsächliches Ergebnis: 0 Zeilen — obwohl es Kunden ohne Tickets gibt.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, die alle Kunden ohne Ticket ausgibt. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [30]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT
c.customerid,
c.firstname,
c.lastname
FROM customers c
LEFT JOIN tickets t ON c.customerid = t.customerid
WHERE t.ticketid IS NULL;
""", con=engine)

engine.dispose()
df_B2

,customerid,firstname,lastname
0,CU0503,Julia,Schmid
1,CU0505,Sandra,Fischer
2,CU0045,Sara,Schmidt
3,CU0504,Kevin,Keller
4,CU0432,Lucas,Wolf
5,CU0259,Max,Keller
6,CU0019,David,Krüger
7,CU0502,Thomas,Berger
8,CU0334,Mia,Krüger
9,CU0270,Marie,Braun


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `cinema.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Die gültigen Werte in `customers.membership_type` sind `Standard`, `Premium`, `Student` und `Senior`. Schreiben Sie eine SQL-Abfrage, die alle Kunden (`customerid`, `firstname`, `lastname`, `membership_type`) mit einem ungültigen Wert in `membership_type` ausgibt — d.h. alle Werte ausser diesen vier. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [31]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT
customerid,
firstname,
lastname,
membership_type
FROM customers
WHERE membership_type NOT IN ('Standard', 'Student', 'Senior');
""", con=engine)

engine.dispose()
df_C1

,customerid,firstname,lastname,membership_type
0,CU0004,Tom,Richter,Premium
1,CU0007,Paul,Bauer,Premium
2,CU0009,David,Richter,Premium
3,CU0013,Lea,Wolf,Premium
4,CU0014,Emma,Keller,Premium
...,...,...,...,...
128,CU0501,Anna,Müller,standard
129,CU0502,Thomas,Berger,STANDARD
130,CU0503,Julia,Schmid,Standart
131,CU0504,Kevin,Keller,premium


### C2 (4 Punkte)

Wie hoch ist der durchschnittliche bezahlte Ticketpreis (`paid_price`) pro Mitgliedschaftstyp (`membership_type`)? Geben Sie `membership_type` und den Durchschnittspreis (`avg_price`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_price`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [32]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT
ROUND(CAST(AVG(paid_price) AS NUMERIC), 2) AS avg_price,
membership_type
FROM customers
INNER JOIN tickets
    ON customers.customerid = tickets.customerid
GROUP BY membership_type;
""", con=engine)

engine.dispose()
df_C2

,avg_price,membership_type
0,15.49,Standard
1,15.34,Senior
2,15.59,Student
3,15.62,Premium


### C3 (4 Punkte)

Welche Kinos haben mehr als 400 Tickets verkauft? Geben Sie `name` und Anzahl Tickets (`ticket_count`) aus, absteigend sortiert nach `ticket_count`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

> [Hier eintragen]

**SQL-Abfrage:**

In [33]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT
cinemas.name,
COUNT(*) AS ticket_count
FROM cinemas

INNER JOIN screenings
    ON cinemas.cinemaid = screenings.cinemaid
INNER JOIN tickets
    ON screenings.screeningid = tickets.screeningid

GROUP BY cinemas.name
HAVING COUNT(*) > 400
ORDER BY ticket_count DESC;

""", con=engine)

engine.dispose()
df_C3

,name,ticket_count
0,Kinok,441
1,Pathé Küchlin,436


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [34]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-10-21 21:07:10
Python Version: 3.12.11
IP Address: 127.0.0.1
-----------------------------------
